In [ ]:
# Install bdsim.  In JupyterLite, piplite checks the local wheel
# (built with 'make wheel') before falling back to PyPI.
# In a regular Jupyter server this is a no-op (piplite is not available).
try:
    import piplite
    await piplite.install('bdsim')
except ImportError:
    pass  # already installed (classic Jupyter)


# bdsim example

We will build a model of a simple dynamic system, a bouncy ball dropped from a certain height.

We first need to import the `bdsim` package

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

import bdsim

and then create an instance of the simulation environment

In [ ]:
sim = bdsim.BDSim()

This does a number of things. Firstly, it finds all the predefined blocks that we can use in our model. 


In [ ]:
sim.blocks()

The ones prefixed by `bdsim.blocks` are shipped with the bdsim package but the Robotics Toolbox and MachineVision Toolbox for Python also define blocks, and if you have them installed they might be found and listed.

Note that the block names shown here are in camel case whereas later we will be referring to with uppercase versions of these names.

This function also evaluates all the options that can be set on the command line, there are lots

as well as being set in the environment and by arguments passed to BDSim.  You can see the current option settings by

In [ ]:
print(sim)

Next up we create an empty block diagram, it's a blank canvas on which we will build our model 

In [ ]:
bd = sim.blockdiagram()

The block diagram `bd` instance holds our model, and the `sim` instance will run our model.  There is a clear separation of data and runtime environment.  Perhaps one day the model can be passed to a different kind of `sim` instance that would run it in real time.


In [ ]:
print(bd)

For our physical system we need to define some constants. We will assume that height is is positive in the upwards direction.

In [ ]:
g = -9.81  # gravity m/s2
e = 0.8  # coefficient of restitution
h0 = 10  # initial height m
v0 = 0  # initial velocity m/s

Now it's time to build our model and it's pretty straightforward, but we have to be thinking in block diagram terms. I imagine a cascade of two integrators, being fed acceleration to produce velocity and height.

We start by creating a block that outputs a constant, the graviational acceleration

In [ ]:
gravity = bd.CONSTANT(g, name="gravity")

Stuff about factory method

In [ ]:
type(gravity)

In [ ]:
print(gravity)

Next we add an integrator that will, soon, have acceleration as input and output velocity

In [ ]:
velocity = bd.INTEGRATOR(name="velocity", x0=0)  # initial height is initial velocity

Then we add another integrator that will, soon, have velocity as input and output height 

In [ ]:
height = bd.INTEGRATOR(name="height", x0=h0)

And we want to see what's going on so we create a SCOPE block to plot the height and velocity.  We have lots of options here, we could plot height and velocity against each other for a phase plot, or plot them against time in separate scopes.

For this example we'll plot them both against time in the same scope plot.  We specify the line styles and where we want the legend.

In [ ]:
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

If we look at the block diagram object now

In [ ]:
print(bd)

We see that it now has 4 blocks.

Now it's time to connect the blocks together by adding some virtual wires.

In [ ]:
bd.connect(gravity, velocity)

  This connects the output of the `gravity` block to the input of the `velocity` block. Remember that each of `gravity` and `velocity` are references to instances of subclasses of the `Block` object.  Block diagrams support fan out, which means that one output can connect to the inputs of many blocks, and this is expressed by listing additional arguments to `connect` -- the first one is the source, the rest are destinations.

  We do a similar thing to wire the output of the first integrator to the input of the second.

In [ ]:
bd.connect(velocity, height)

Next we connect the scopes, but the syntax will be a little different this time because the scope we created has two inputs

In [ ]:
print(scope)

Note about naming

and we see that a scope block has 2 inputs.

one for each of the signals we will plot against time.  When we do our connecting we need to specify which input to connect to.  The earlier examples were easy because the integrators have just one input and one output.  The syntax is very pythonic

In [ ]:
bd.connect(velocity, scope[0])
bd.connect(height, scope[1])

which connects the output of `velocity` to input 0 of the scope, and the output of `height` to input 1 of the scope.

The next is to compile the system.  This checks the connections are valid, that there are no ?? loops, and determines what order the blocks should be executed in.

In [ ]:
bd.compile()
sim.report(bd)

We can also see where the block names have been assigned by us in the code (eg. `gravity`, `height`, `velocity`) and we've left them to get auto generated names (eg. `scope.0`).  The way to read this table is that height has 1 input which comes from the `velocity` block and is a NumPy array of shape (1,) and floating point.  `scope.0` has two inputs, one from `velocity` and one from `height`.

Now it's time to run our model.  We'll see what happens over the first 2 seconds.

In [ ]:
out = sim.run(bd, T=2)

Not surprisingly it falls.  The height of ball decreases quadratically over time, while its velocity decreases linearly.  Our ball passes through the ground at around $t=1.4$s and keeps going into negative height territory -- that's because we haven't modeled the ground and what happens when the ball hits it.

We first need to do determine when the ball hits the ground, and then do something about it.  We'll tackle the second part first.  At impact we want the ball's velocity to change sign, to be bouncing upwards.  But we also want to scale it so that the ball doesn't bounce forever, and to model the energy loss (as heat) during the impact. This  scale factor is called the *coefficient of restitution*. It is 1 for a lossless bounce, and 0 for a lump of putty.

At impact we will modify the state of the first integrator.

In [ ]:
def impact(x):
    velocity.x *= -e  # reverse velocity and apply restitution

Next we need to determine when to invoke the `impact` function.  It must be called when the ball hits the ground, that is, when its height crosses from being positive to being negative. 

In [ ]:

ground = bd.EVENT("-", impact)


We do this with an EVENT block that invokes a specified function when its input signal crosses zero.  In this case, by providing the "-" argument (a negative going transition) the `impact` function will be called when the height changes from positive to negative.  The integrator normally takes quite large steps when integrating such simple dynamics, but it essentially performs a root search to hone in on the exact moment of crossing.

Now we need to wire this block into our system.

In [ ]:
bd.connect(height, ground)

and then incrementally recompile our system

In [ ]:
bd.compile()

In [ ]:
sim.report(bd)

In [ ]:
# bd.compile()
# sim.report(bd)

In [ ]:
out = sim.run(bd, T=10)

explore output object, pickle json
save a plot as PDF
save a movie and view it